# Data Day Tutorial: Structured data — cohort extraction

In the lecture we've discussed that, for imaging projects and our use-case **OsteoHand**, structured data is what makes the imaging model possible.
In this tutorial we'll use it to **find** the cohort and **link** it to the hand X-rays via an
**accession number**. We'll also explore the **DXA T-score** metric as a possible reference label for whoever does the labelling downstream.

### What we will cover

| Part | Topic | Time |
|------|-------|------|
| 0 | Setup | 2 min |
| 1 | Load & explore the tables (including the imaging table) | 5 min |
| 2 | Cohort step 1 — age & the DXA T-score | 6 min |
| 3 | Cohort step 2 — link the hand X-ray (12-month window) | 6 min |
| 4 | Data quality — grain, missing accessions & dates | 5 min |
| 5 | Export the hand-off | 3 min |
| Advanced | The same query in SQL (DuckDB) & window sensitivity | if time allows |

<font color="red">Cells under red text are part of the exercises and may not run properly until you add some code.</font>

<font color="green">If you'd rather just run the cells and follow along, click the **"Show code"** toggle under the green text to show the worked solution, then run it.</font>

---
## Part 0: Setup

Run the cell below to install what we need. You can read ahead while it runs.

In [ ]:
# -q keeps it quiet. gdown = download from Google Drive; duckdb is used only in the Advanced part.
!pip install -q gdown duckdb

import pandas as pd
import numpy as np
from datetime import timedelta

pd.set_option('display.max_columns', None)
print("Setup complete — pandas", pd.__version__)

---
## Part 1: Load & explore the tables

Download the synthetic EHR extract (generated with **Synthea**) into this Colab session.

In [ ]:
import gdown, os
file_id = '1yGbk94tbMZd8cOCBtXWsb22Wa0pJwOhk'
url = f'https://drive.google.com/uc?export=download&id={file_id}'
output = 'synthetic-medical-dataset.zip'
if not os.path.exists(output):
    print("Downloading dataset..."); gdown.download(url, output, quiet=False)
else:
    print("Already downloaded.")
!unzip -qq -o synthetic-medical-dataset.zip -d data/
print("Dataset ready in ./data/")

The dataset is split across several CSVs, one per record type, similar to an SDE:

| Table | What it holds |
|-------|---------------|
| `patients` | one row per patient (demographics) |
| `observations` | measurements & results |
| `conditions` | coded diagnoses |
| `procedures` | coded procedures |
| `encounters` | visits |

In [ ]:
BASE = 'data/archive-2/'
patients     = pd.read_csv(BASE + 'patients.csv',     parse_dates=['birthdate', 'deathdate'])
encounters   = pd.read_csv(BASE + 'encounters.csv',   parse_dates=['DATE'])
observations = pd.read_csv(BASE + 'observations.csv', parse_dates=['DATE'])
conditions   = pd.read_csv(BASE + 'conditions.csv',   parse_dates=['START'])
procedures   = pd.read_csv(BASE + 'procedures.csv',   parse_dates=['DATE'])

patients = patients.rename(columns={'patient': 'PATIENT'})  # standardise the join key across tables

_rng = np.random.default_rng(0)
_dxa_mask = observations['DESCRIPTION'].str.contains('DXA', case=False, na=False)
observations.loc[_dxa_mask, 'VALUE'] = np.round(
    _rng.normal(-1.3, 1.0, _dxa_mask.sum()), 2).clip(-4.0, 1.5).astype(str)

print("Loaded:")
for name, df in [('patients',patients),('encounters',encounters),('observations',observations),
                 ('conditions',conditions),('procedures',procedures)]:
    print(f"  {name:<13} {len(df):>7,} rows")

### Add the imaging table

Unfortunately our Synthea dataset doesn't contain imaging data with hand radiographs with relevant accession numbers. So here we simulate an `imaging_orders` table to treat as a
replacement for the extract you'd pull from the **RIS** (radiology information system) tables in your SDE. It joins to the other tables on `PATIENT`, just like a real one would.

Just run the cell to create it. (Expand the title if you want to see how it's built in pandas)

In [ ]:
#@title ▶ Run me — create the (simulated) imaging_orders table {display-mode: "form"}
rng = np.random.default_rng(42)   # fixed seed -> everyone gets the same data
def accession(): return f"ACC{rng.integers(10_000_000, 99_999_999)}"   # ACC######## (same shape as the other tutorials)
def study_uid(): return f"1.2.840.{rng.integers(1000,9999)}.{rng.integers(100_000,999_999)}"

# We anchor each simulated hand X-ray near that patient's DXA date so some — not all — fall within 12 months.
_dxa = observations[observations['DESCRIPTION'].str.contains('DXA', case=False, na=False)].copy()
_dxa['t'] = pd.to_numeric(_dxa['VALUE'], errors='coerce')
_dxa_first = _dxa.dropna(subset=['t']).sort_values('DATE').groupby('PATIENT', as_index=False).first()

rows = []
# (a) ~70% of DXA patients also get a hand X-ray (NOT everyone), a few with a repeat xray
for _, r in _dxa_first.iterrows():
    if rng.random() < 0.70:
        for _ in range(int(rng.choice([1,1,1,2]))):
            offset = int(rng.normal(0, 250))   # most within ~8 months; tails fall OUTSIDE 12 months
            rows.append(dict(PATIENT=r['PATIENT'], accession_number=accession(),
                             study_datetime=r['DATE'] + pd.Timedelta(days=offset),
                             modality='XR', body_site='Hand',
                             study_instance_uid=study_uid(), laterality=rng.choice(['L','R'])))
# (b) noise: hand X-rays for some patients with NO DXA (must be excluded later — no label)
_non = patients.loc[~patients['PATIENT'].isin(_dxa_first['PATIENT']), 'PATIENT']
for pid in rng.choice(_non, size=min(120, len(_non)), replace=False):
    rows.append(dict(PATIENT=pid, accession_number=accession(),
                     study_datetime=pd.Timestamp('2015-06-01') + pd.Timedelta(days=int(rng.integers(0,2500))),
                     modality='XR', body_site='Hand',
                     study_instance_uid=study_uid(), laterality=rng.choice(['L','R'])))

imaging_orders = pd.DataFrame(rows)
# realistic data-quality issue: ~8% of orders have a MISSING accession number (we deal with this in Part 4)
imaging_orders.loc[imaging_orders.sample(frac=0.08, random_state=1).index, 'accession_number'] = np.nan
print(f"imaging_orders: {len(imaging_orders):,} hand X-ray studies for "
      f"{imaging_orders['PATIENT'].nunique():,} patients")
imaging_orders.head()

### Inspect the schemas

Look at the shape and columns of each table before we query anything.

In [ ]:
for name, df in [('patients',patients), ('observations',observations), ('imaging_orders',imaging_orders)]:
    print(f"\n{'='*64}\n  {name.upper()}  —  {df.shape[0]:,} rows | {df.shape[1]} columns")
    print(f"  Columns: {list(df.columns)}")
    display(df.head(3))

### Exercise 1.1 — Quick exploration

1. How many unique patients are in `patients`?
2. How many `observations` are DXA results? *(hint: the `DESCRIPTION` column)*
3. How many `imaging_orders` have a missing accession number? *(hint: `.isna()`)*

<font color="red">If you'd like to code here are some starting points you may find helpful:</font>

In [ ]:
# 1. unique patients -> .nunique()
# n = patients['PATIENT']._____

# 2. DXA rows -> .str.contains('DXA', case=False, na=False) then .sum()
# is_dxa = observations['DESCRIPTION']._____

# 3. missing accessions -> .isna().sum()
# missing = imaging_orders['accession_number']._____

<font color="green">Worked solution:</font>

In [ ]:
#@title Click to reveal code {display-mode: "form"}
print("Unique patients:", patients['PATIENT'].nunique())
print("DXA observations:", int(observations['DESCRIPTION'].str.contains('DXA', case=False, na=False).sum()))
print("Imaging orders with a missing accession:", int(imaging_orders['accession_number'].isna().sum()))

---
## Part 2: Cohort step 1 — age & the DXA T-score

Our cohort: **adults 50+** with a **DXA scan** *and* a **hand X-ray** within **12 months** of each other.
First we will extract **age** and the **DXA**, Part 3 links us to the hand X-ray.

A **DXA scan** measures bone density. Its main output is the **T-score**, which compares a person's bone density to that of a healthy young adult. A T-score of 0 is average and the more negative the score, the weaker the bone.

The standard thresholds split the T-score into three categories:

| T-score | Category | Meaning |
|---|---|---|
| **≥ −1.0** | Normal | bone density in the healthy range |
| **> −2.5 and < −1.0** | Osteopenia | lower than normal (sometimes called low bone mass) |
| **≤ −2.5** | Osteoporosis | low enough to indicate osteoporosis |

We extract the T-score and tag each patient with its category, it may not be used to label but we carry it through so it's available downstream.  (the labelling itself is handled separately)

### Step 2.1 — Patients aged 50 and over

In [ ]:
index_date = pd.Timestamp('2024-12-31')
patients['age'] = (index_date - patients['birthdate']).dt.days // 365
aged_50_plus = patients[patients['age'] >= 50]
print(f"Patients aged 50+: {len(aged_50_plus):,} of {len(patients):,}")
aged_50_plus[['PATIENT','birthdate','age','gender']].head(3)

### Step 2.2 — DXA T-score


* **T ≤ −2.5 → Osteoporosis**
* **−2.5 < T < −1.0 → Osteopenia**
* **T ≥ −1.0 → Normal**

A patient may have several DXA scans, so we keep **one per patient** (their earliest) — one stable value and label each. We also convert the T-score to a number, turning any non-numeric entries into blanks. This dataset is clean, but real EHR data often isn't.

In [ ]:
dxa = observations[observations['DESCRIPTION'].str.contains('DXA', case=False, na=False)].copy()
dxa['t_score'] = pd.to_numeric(dxa['VALUE'], errors='coerce')
dxa = dxa.sort_values('DATE')
dxa_first = dxa.dropna(subset=['t_score']).groupby('PATIENT', as_index=False).first()

# 3-class band, carried as a column (not converted to a 0/1 target)
dxa_first['label'] = np.select(
    [dxa_first['t_score'] <= -2.5, dxa_first['t_score'] < -1.0],
    ['Osteoporosis', 'Osteopenia'], default='Normal')

order = ['Osteoporosis', 'Osteopenia', 'Normal']
print(f"Patients with a DXA T-score: {len(dxa_first):,}")
print(dxa_first['label'].value_counts().reindex(order))
dxa_first[['PATIENT','DATE','t_score','label']].head()

### Exercise 2.3 — Bone-density band distribution

What **proportion** of DXA'd patients fall in each band?

<font color="red">If you'd like to code here are some starting points you may find helpful:</font>

In [ ]:
# .value_counts(normalize=True) gives proportions instead of counts
# dist = dxa_first['label']._____

<font color="green">Worked solution:</font>

In [ ]:
#@title Click to reveal code {display-mode: "form"}
order = ['Osteoporosis', 'Osteopenia', 'Normal']
print((dxa_first['label'].value_counts(normalize=True).reindex(order, fill_value=0) * 100).round(1).astype(str) + ' %')

---
## Part 3: Cohort step 2 — link the hand X-ray

Now we join the hand X-rays (from `imaging_orders`) to the DXA patients and apply the **12-month window**.
The `merge` is an **inner join**, so any hand X-ray for a patient *without* a DXA drops out automatically (i.e. no DXA = not in the study).

We can see our cohort reduce with each join/filter.

In [ ]:
linked = imaging_orders.merge(dxa_first[['PATIENT','DATE','t_score','label']], on='PATIENT')  # drops no-DXA noise
linked = linked.merge(patients[['PATIENT','age','gender']], on='PATIENT')
linked['gap_days'] = (linked['study_datetime'] - linked['DATE']).abs().dt.days

print(f"Hand X-rays for DXA patients:        {len(linked):,}")
cohort = linked[linked['age'] >= 50]
print(f"... aged 50+:                        {len(cohort):,}")
cohort = cohort[cohort['gap_days'] <= 365]
print(f"... AND within 12 months of the DXA: {len(cohort):,}  ({cohort['PATIENT'].nunique():,} unique patients)")
cohort.head()

### Exercise 3.1 — How much work is the window doing?

How many hand X-rays (for DXA'd, age 50+ patients) are excluded **only because** they fall *outside* the
12-month window?

<font color="red">If you'd like to code here are some starting points you may find helpful:</font>

In [ ]:
# eligible = linked[linked['age'] >= 50]
# outside  = eligible[eligible['gap_days'] > 365]
# print(len(outside))

<font color="green">Worked solution:</font>

In [ ]:
#@title Click to reveal code {display-mode: "form"}
eligible = linked[linked['age'] >= 50]
outside  = eligible[eligible['gap_days'] > 365]
print(f"{len(outside):,} hand X-rays excluded only by the 12-month window "
      f"({len(outside)/len(eligible):.1%} of otherwise-eligible studies).")

---
## Part 4: Data quality

Structured data extracts have their own failure points. These are some examples you may want to consider when identifying imaging cohorts for AI model development.

### 4.1 — Handling multiple X-Rays per patient

A patient with **two** qualifying hand X-rays will appear as **two rows**.

We must choose our level of detail:

* **one row per patient** — for counts/prevalence
* **one row per study (accession)** — i.e. don't drop valid images

In [ ]:
xrays = cohort['PATIENT'].value_counts()
multiples = xrays[xrays > 1]
print(f"Patients with more than one qualifying hand X-ray: {len(multiples)}")
if len(multiples):
    display(cohort[cohort['PATIENT'] == multiples.index[0]][['PATIENT','accession_number','study_datetime','gap_days','label']])

patient_level = cohort.sort_values('gap_days').drop_duplicates('PATIENT')  # keep the xray closest to the DXA
print(f"Patient-level rows: {len(patient_level):,}   |   Study-level rows: {len(cohort):,}")

### 4.2 — Missing accession numbers

An imaging order with **no accession number** can't be traced to an actual image — so it's useless for the
imaging hand-off, however valid the patient is. Find them and drop them.

In [ ]:
n_missing = cohort['accession_number'].isna().sum()
print(f"Cohort studies with a missing accession: {n_missing}")
cohort = cohort[cohort['accession_number'].notna()].copy()
print(f"Usable cohort studies after dropping them: {len(cohort):,}")

### Exercise 4.4 — Discussion

Consider the resulting cohort in the context of the imaging model you are planning to train. Would you keep **one x-ray per patient** or **every qualifying x-ray**?
And what must you keep either way?

<details>
<summary><b>Click to show answer</b></summary>
Keep <b>every qualifying x-ray</b> — more images is more training data. But you <b>must</b> retain a per-patient
ID so the imaging team can split train/test <b>by patient</b>, not by image. Otherwise the same patient's x-rays
land in both splits and the model can memorise the patient — the same <i>data-leakage</i> risk the text tutorial
flags for duplicated reports.
</details>

---
## Part 5: Export the final eligible cohort

Our end goal: a list of the eligible patients and the **accession numbers** for their hand X-rays, so that the images can be pulled from **PACS** for downstream labelling and imaging model development.

Each row is one **study**, with its `t_score` and bone-density band alongside the accession. We also **pseudonymise** the patient ID so a patient's studies can still be grouped (i.e. for the train/test split) without needing to expose the real ID.

In [ ]:
id_map = {pid: f"SID{idx:05d}" for idx, pid in enumerate(cohort['PATIENT'].unique())}
handoff = cohort.copy()
handoff['study_id'] = handoff['PATIENT'].map(id_map)
handoff = handoff[['accession_number','study_instance_uid','t_score','label','age','gender','study_id']]
handoff.to_csv('osteohand_cohort_handoff.csv', index=False)

print(f"Hand-off rows (studies): {len(handoff):,}")
print(handoff['label'].value_counts().to_dict())
handoff.head()

**Anonymisation note.**

We kept the **accession numbers** (in order to fetch the X-Rays from PACS)
but replaced the patient ID with a `study_id`, so the real mapping would stay inside the SDE and the images will get
de-identified downstream. Here we also leave the **T-score and bone-density band** as a reference label however labelling may be handled separately (i.e checking report text, clinician/expert review).


---
## Advanced (if time allows)

### A — The same query, in SQL

In a real project this would be run this against your SDE in **SQL**. **DuckDB** simulates this for us and runs real SQL directly on our
dataframes (no server needed) so you can see the equivalence.

Check the distinct patient count should match what we did in pandas.

<font color="red">If you'd like to code here are some starting points you may find helpful:</font>

In [ ]:
# Fill in the blanks to rebuild the cohort in SQL, then compare the count to pandas.
import duckdb
con = duckdb.connect()
con.register('patients', patients); con.register('dxa', dxa_first); con.register('imaging', imaging_orders)

sql = con.execute('''
    SELECT count(DISTINCT p.PATIENT) AS n_patients
    FROM patients p
    JOIN dxa     d ON d.PATIENT = p.PATIENT
    JOIN imaging i ON ____________________            -- TODO: join imaging to patients on PATIENT
    WHERE p.age >= ____                                -- TODO: age filter
      AND abs(date_diff('day', d.DATE, i.study_datetime)) <= ____   -- TODO: the 12-month window, in days
''').df()
print("DuckDB patients:", int(sql['n_patients'][0]))

<font color="green">Worked solution:</font>

In [ ]:
#@title Click to reveal code {display-mode: "form"}
import duckdb
con = duckdb.connect()
con.register('patients', patients); con.register('dxa', dxa_first); con.register('imaging', imaging_orders)
sql = con.execute('''
    SELECT count(DISTINCT p.PATIENT) AS n_patients
    FROM patients p
    JOIN dxa     d ON d.PATIENT = p.PATIENT
    JOIN imaging i ON i.PATIENT = p.PATIENT
    WHERE p.age >= 50
      AND abs(date_diff('day', d.DATE, i.study_datetime)) <= 365
''').df()
pandas_n = linked[(linked['age'] >= 50) & (linked['gap_days'] <= 365)]['PATIENT'].nunique()
print(f"DuckDB: {int(sql['n_patients'][0]):,}  |  pandas: {pandas_n:,}  |  match: {int(sql['n_patients'][0]) == pandas_n}")

### B — How sensitive is the cohort to the window?

<font color="red">If you'd like to code here are some starting points you may find helpful:</font>

In [ ]:
# Change window_months and re-run. How does the cohort size move?
window_months = 12        # <- try 3, 6, 24 ...

days = int(window_months / 12 * 365)
n = linked[(linked['age'] >= 50) & (linked['gap_days'] <= days)]['PATIENT'].nunique()
print(f"{window_months} months  ->  {n:,} patients")

<font color="green">Worked solution:</font>

In [ ]:
#@title Click to reveal code {display-mode: "form"}
for window_months in [3, 6, 12, 24]:
    days = int(window_months / 12 * 365)
    n = linked[(linked['age'] >= 50) & (linked['gap_days'] <= days)]['PATIENT'].nunique()
    print(f"{window_months:>2} months  ->  {n:,} patients")

---
## Recapping

From raw multi-table EHR data we produced a linked cohort:

* **Find** — age 50+ filter + the 12-month temporal join
* **Link** — `PATIENT -> imaging order -> accession number`
* **Keep** — the DXA T-score & bone-density band, as a reference label
* **Data Quality** — considered handling multiple scans/patient, dropped studies with no accession number
* **Output** — minimised & pseudonymised, but kept the accession numbers

You now have the information needed to identify your images for AI model development.